# 10 · Calibration, final-model rule and the test run (Step 7)

**Order matters here, and the files show it:**

1. `final_fit`: every family is refitted on all training rows available at 2013-09-01 (5 seeds, fixed epochs = mean CV best epoch). Scores need no labels.
2. `calibrate`: the Phase 1 **Sep → Oct** rule for every model (fit on September, check ECE on October, adopt a recalibrator only if it beats raw by > 0.005), extended with temperature scaling (Guo et al., 2017) and causal rolling temperature; thresholds frozen on Sep–Oct; the **final model chosen by CV**. No test label is read.
3. `final` (only with `--final`, refused while the protocol has uncommitted changes): the one test evaluation.

This notebook only reads the saved results.

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
for p in (ROOT / 'src', ROOT.parent / 'phase1' / 'src'):
    sys.path.insert(0, str(p))

import aerocast_dl  # torch before XGBoost (OpenMP)
import numpy as np, pandas as pd
from IPython.display import Image, display
from aerocast_dl import config, pipeline
R, F = config.RESULTS_DIR, config.FIGURES_DIR
pd.set_option('display.precision', 4)


## 1. Calibrator selection on October

In [ ]:
cal = pd.read_csv(R / 'calibrator_selection.csv')
sel = json.loads((R / 'selection.json').read_text())
print('chosen:', sel['calibrator'])
cal.pivot(index='calibrator', columns='model', values='check_ece').round(4)

## 2. The final-model rule (cross-validation only)

Highest mean rolling-CV AP as deployed; within 0.005 of the best, the model with fewer trainable parameters.

In [ ]:
pd.DataFrame({'cv_pr_auc': sel['cv_pr_auc'], 'parameters': sel['complexity']}).sort_values('cv_pr_auc', ascending=False).assign(within_0_005=lambda d: d.index.isin(sel['within_0.005']))

In [ ]:
print('final model:', sel['final_model'])

## 3. Test (Nov–Dec), each model with its rule-selected calibrator

The paired day-block bootstrap compares each model with Phase 1 LogReg on the same days; an interval that contains 0 establishes neither a difference nor equivalence.

In [ ]:
m = pd.read_csv(R / 'test_metrics.csv')
head = m[(m.split == 'test') & m.rule_selected].set_index('label')
head[['calibration', 'pr_auc', 'roc_auc', 'brier', 'ece', 'precision_at_thr', 'recall_at_thr']]

In [ ]:
boot = json.loads((R / 'test_bootstrap.json').read_text())
pd.DataFrame({k: {'AP diff vs LogReg': v['pr_auc']['diff'], 'CI low': v['pr_auc']['ci_lo'], 'CI high': v['pr_auc']['ci_hi']} for k, v in boot.items()}).T

In [ ]:
pol = pd.read_csv(R / 'test_policies.csv')
pol[pol.policy == 'daily top 10%'].set_index('label')[['precision', 'recall', 'lift', 'f1', 'accuracy']]

In [ ]:
m[m.rule_selected & m.split.isin(['test_m11', 'test_m12'])].pivot(index='label', columns='split', values='ece').round(4)

In [ ]:
display(Image(F / 'reliability_test.png', width=420))